# Script to update metadata for any dataset

Check metadata for old sequences to see if anything has been added

Databases: GISAID, Andersen, NCBI Virus

In [1]:
# Housekeeping


import os
import glob 
import pandas as pd
import xml.etree.ElementTree as ET
import requests
import time
import numpy as np
import dateutil 
from datetime import datetime
from collections import defaultdict 
import importlib
import utils  
importlib.reload(utils)
from utils import * 

# Make sure you have the correct paths

# Dates
start_date = "11-01-2023"
end_date = "05-14-2025"
date_range = start_date + "--" + end_date
update_date = "06-11-2025"

references = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/references/"
os.chdir(references)
states_ref = pd.read_csv("states_ref.csv")

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
downloads_gisaid = "C:/Users/maksiaevai.NCBI_NT/Downloads/" # MUST be downloads folder for gisaid
# downloads_gisaid = home + "GISAID/downloads/" + "2024-01-01--2025-05-14_North_America/"
downloads_ncbi_virus = home + "NCBI_Virus/downloads/" # + date_range + "/"

genotype = "B3.13"
genotype_underscored = genotype.replace(".", "_")
originals = home + "Combinations/GISAID_Andersen_NCBI_Virus/" + date_range + "_" + genotype_underscored + "/"

complete = home + "Combinations/GISAID_Andersen_NCBI_Virus/" + date_range + "_" + genotype_underscored + "/updated_" + update_date + "/"

os.chdir(originals)

## Original Files

In [2]:
# Function to prepare dataframes
def fasta_df_og(file_name, state_ref):

    fasta = pd.DataFrame()
    headers = []
    isolate_ids = []
    isolate_names = []
    subtypes = []
    # segments = []
    collection_dates = []
    sequences = []
    host_types = []
    species = []
    identifiers = []
    genotypes = []
    with open(file_name) as f:
        lines = f.readlines()
        for num, line in enumerate(lines):
            # print(line)
            if line[0] == ">": # If it's a header
                if line[1:].strip() not in headers: # And the previous line is not a header we've seen before
                    header = line[1:].strip() # Remove the ">"
                    # print(header)
                    split_header = header.split("|")
                    if len(header.split("|")) > 6:
                        identifier = header.split("|")[0]
                        identifiers.append(identifier)
                    else:
                        identifiers.append("unknown")
                    split_first_header = split_header[-6].split("/")
                    # print(split_first_header)
                    # print(split_header)
                    headers.append(header) 
                    isolate_ids.append(split_first_header[3])
                    isolate_names.append(split_header[-6]) # We'll need to extract data from this too
                    # print(split_header[2].split("_")[-1])
                    subtypes.append(split_header[-5])  # Get only H5N1
                    genotypes.append(split_header[-1])
                    # segments.append(split_header[-3].split("_")[-1])
                    host_types.append(split_header[-2])
                    species.append(split_first_header[1])
                    # if split_header[4] == "2024-01-01":
                    #     collection_dates.append("2024") # No samples were collected 1/1/2024, these are all unknown 
                    # elif split_header[4] == "2025-01-01":
                    #     collection_dates.append("2025")
                    # else: 
                    collection_dates.append(split_header[-3].split("_")[-1])
                    if num < len(lines): # If we're not at the last line
                        # for i, l in enumerate(lines[num + 1:]):
                        i = num
                        sequence = ""
                        # print(lines[i])
                        # print(lines[i + 1])
                        while i < len(lines) - 1 and lines[i + 1][0] != ">": # While the next line is part of a sequence
                            sequence = sequence + lines[i + 1].strip()
                            i += 1
                        sequences.append(sequence) # Add next line to sequences
        f.close()

    # Create columns for data frame 
    fasta["Header"] = headers
    fasta["Isolate_Id"] = isolate_ids
    fasta["Isolate_Name"] = isolate_names
    fasta["Subtype"] = subtypes
    # fasta["Segment"] = segments
    # Geo_Location is more complicated
    fasta["Geo_Location"] = fasta["Header"].apply(lambda x: state_ref.loc[state_ref["Abbreviation"] == x.split("/")[2], 'Country'].iloc[0] + "-" + x.split("/")[2] if x.split("/")[2] in state_ref["Abbreviation"].values else state_ref.loc[state_ref["State"] == x.split("/")[2].replace("_", " "), 'Country'].iloc[0] + "-" + state_ref.loc[state_ref["State"] == x.split("/")[2].replace("_", " "), 'Abbreviation'].iloc[0] if x.split("/")[2].replace("_", " ") in state_ref["State"].values else x.split("/")[2].replace(": ", "-"))
    fasta["Date Collected"] = collection_dates
    fasta["Species"] = species
    fasta["Host_Type"] = host_types
    fasta["Genotype"] = genotypes
    fasta["Sequence"] = sequences
    if len(identifiers) == len(fasta):
        fasta["Identifier"] = identifiers
    
    return fasta

original_fasta_dfs = {}

for dirpath, dirs, files in os.walk(originals):
    for file in files:
        file_name = os.path.join(dirpath, file)
        if ".fasta" in file_name:
            fasta_file = fasta_df_og(file_name, states_ref)
            original_fasta_dfs[file_name] = fasta_file
    break 

## GISAID

In [ ]:
# Get data from GISAID

username = input("Username: ")
password = input("Password: ")
browser = input("Browser: ")
sleep_time = input("Seconds to sleep in between clicks: ")
continent = input("Continent(s) separated by commas: ")
start_date = dateutil.parser.parse(start_date).strftime("%Y-%m-%d") # Make sure date is in correct format
end_date = dateutil.parser.parse(end_date).strftime("%Y-%m-%d")

open_gisaid(username, password, browser, sleep_time, continent, start_date, end_date)

In [ ]:
# Get downloaded GISAID data

all_metadata_files = []
all_fasta_files = []

# Grab files
for dirpath, dirs, files in os.walk(downloads_gisaid):
    for file in files:
        file_name = os.path.join(dirpath, file)
        # print(file_name)
        if ".xls" in file_name:
            metadata = pd.read_excel(file_name, engine="xlrd")
            all_metadata_files.append(metadata)
        if ".fasta" in file_name:
            fasta_file = fasta_df(file_name, states_ref) # Convert fasta file to dataframe
            all_fasta_files.append(fasta_file)

In [ ]:
# Search for dates and states based on isolate

for key in original_fasta_dfs:
    og_df = original_fasta_dfs[key]
    print(og_df)
    og_df["Unknown_States"] = og_df["Geo_Location"].apply(lambda x: 1 if x == "USA" else 0)
    og_df["Unknown_Dates"] = og_df["Date Collected"].apply(lambda x: 1 if x == "2022-01-01" or x == "2023-01-01" or x == "2024-01-01" or x == "2025-01-01" else 0)
    og_df["Update_Needed"] = og_df["Unknown_States"] + og_df["Unknown_Dates"]
    # og_df["Identifier"] = ""
    og_df_update_needed = og_df[og_df["Update_Needed"] > 0]
    for isolate in og_df_update_needed["Isolate_Id"].values:
        # print(isolate)
        for new_df in all_fasta_files:
            if isolate in new_df["Isolate_Id"].values:
                # print(isolate)
                og_df_update_needed.at[og_df_update_needed[og_df_update_needed["Isolate_Id"] == isolate].index[0], "Geo_Location"] = new_df.loc[new_df[new_df["Isolate_Id"] == isolate].index[0], "Geo_Location"]
                og_df_update_needed.at[og_df_update_needed[og_df_update_needed["Isolate_Id"] == isolate].index[0], "Date Collected"] = new_df.loc[new_df[new_df["Isolate_Id"] == isolate].index[0], "Date Collected"]
                og_df_update_needed.at[og_df_update_needed[og_df_update_needed["Isolate_Id"] == isolate].index[0], "Identifier"] = new_df.loc[new_df[new_df["Isolate_Id"] == isolate].index[0], "Identifier"] # .split("|")[0]

    og_df_update_needed = og_df_update_needed.drop_duplicates(subset="Header")
    print(og_df_update_needed)

    # new_df = og_df.merge(og_df_update_needed, how="left")
    new_df = og_df.set_index('Header')
    new_df.update(og_df_update_needed.set_index('Header'))
    new_df = new_df.reset_index()
    # new_df = pd.concat([og_df_update_needed, og_df]).drop_duplicates(['Isolate_Id'], keep="last")
    print(new_df)
    # break 

    original_fasta_dfs[key] = new_df

    

                                                 Header     Isolate_Id  \
0     A/AMERICAN_BLACK_DUCK/USA/25-002460-001/2025|H...  25-002460-001   
1     A/AMERICAN_BLACK_DUCK/USA/25-002634-002/2025|H...  25-002634-002   
2     A/American_Crow/BC/AIVPHL-2662/2024|H5N1|Canad...    AIVPHL-2662   
3     A/American_Crow/BC/AIVPHL-2850/2024|H5N1|Canad...    AIVPHL-2850   
4     A/American_Kestrel/BC/AIVPHL-2866/2024|H5N1|Ca...    AIVPHL-2866   
...                                                 ...            ...   
2235  A/wood_duck/Tennessee/79/2024|H5N1|USA-TN|2024...             79   
2236  A/wood_duck/USA/003927-016/2025|H5N1|USA|2025|...     003927-016   
2237  A/wood_duck/USA/005011-001/2025|H5N1|USA|2025|...     005011-001   
2238  A/wood_duck/USA/007113-001/2025|H5N1|USA|2025|...     007113-001   
2239  A/wood_duck/USA/038225-003/2024|H5N1|USA|2024|...     038225-003   

                                      Isolate_Name Subtype Geo_Location  \
0     A/AMERICAN_BLACK_DUCK/USA/25-0

## Andersen

In [ ]:
# Read metadata

os.chdir(home)
metadata_normalized = pd.read_csv("metadata_normalized.tsv", delimiter="\t") # Collection dates

metadata_folder = home + "Andersen/avian-influenza/metadata/"
os.chdir(metadata_folder)

metadata = pd.read_csv("SraRunTable_automated.csv") # Everything else

print(len(metadata))

# Merge with metadata_normalized
metadata = metadata.merge(metadata_normalized, how="outer")
print(metadata.columns)

# # Find only >= last date using Release Date from metadata 
# metadata["ReleaseDate"] = metadata["ReleaseDate"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
# metadata = metadata[metadata["ReleaseDate"] >= dateutil.parser.parse(start_date).strftime("%Y-%m-%d")]
# # Find only <= update date using Release Date from metadata
# metadata = metadata[metadata["ReleaseDate"] <= dateutil.parser.parse(end_date).strftime("%Y-%m-%d")]

print(len(metadata)) 
display(metadata)

9697
Index(['Run', 'Assay Type', 'AvgSpotLen', 'Bases', 'BioProject', 'BioSample',
       'BioSampleModel', 'Bytes', 'Center Name', 'Collection_Date', 'Consent',
       'DATASTORE filetype', 'DATASTORE provider', 'DATASTORE region',
       'Experiment', 'geo_loc_name_country', 'geo_loc_name_country_continent',
       'geo_loc_name', 'Host', 'Instrument', 'isolate', 'Library Name',
       'LibraryLayout', 'LibrarySelection', 'LibrarySource', 'Organism',
       'Platform', 'ReleaseDate', 'create_date', 'version', 'Sample Name',
       'SRA Study', 'serotype', 'isolation_source', 'BioSample Accession',
       'is_retracted', 'retraction_detection_date_utc'],
      dtype='object')
19050


,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,ReleaseDate,create_date,version,Sample Name,SRA Study,serotype,isolation_source,BioSample Accession,is_retracted,retraction_detection_date_utc
0,SRR24839058,AMPLICON,230.77,32942716,PRJNA980729,SAMN35647642,Viral,18373327,United States Department of Agriculture,missing,...,2023-06-30 00:46:13,2023-06-07 02:01:47,1,22-005893-001,SRP441379,H5N1,",",SRS17903639,False,NaN
1,SRR24839058,AMPLICON,230.77,32942716,PRJNA980729,SAMN35647642,Viral,18373327,United States Department of Agriculture,missing,...,2023-06-30 00:46:13,2023-06-07 02:01:47,1,22-005893-001,SRP441379,H5N1,"Swab, Tracheal",SRS17903639,False,NaN
2,SRR24839059,AMPLICON,207.54,53993591,PRJNA980729,SAMN35647620,Viral,29609119,United States Department of Agriculture,2022-02-08,...,2023-06-30 00:46:13,2023-06-07 02:01:22,1,AH0210318,SRP441379,H5N1,",/",SRS17903636,False,NaN
3,SRR24839059,AMPLICON,207.54,53993591,PRJNA980729,SAMN35647620,Viral,29609119,United States Department of Agriculture,2022-02-08,...,2023-06-30 00:46:13,2023-06-07 02:01:22,1,AH0210318,SRP441379,H5N1,"Swab Pool, Cloacal/Oropharyngeal",SRS17903636,False,NaN
4,SRR24839060,AMPLICON,201.66,21703662,PRJNA980729,SAMN35647621,Viral,10716272,United States Department of Agriculture,2022-02-17,...,2023-06-30 00:46:13,2023-06-07 02:01:28,1,22-005158-001,SRP441379,H5N1,NaN,SRS17903631,False,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19045,SRR33830446,WGS,148.77,95232771,PRJNA1102327,SAMN48895426,Viral,37952018,USDA-NVSL,2025,...,2025-06-06 00:57:17,2025-06-04 14:24:56,1,25-015677-004,SRP503016,NaN,"MILK, BULK TANK",SRS25266464,False,NaN
19046,SRR33830447,WGS,148.43,106145246,PRJNA1102327,SAMN48895425,Viral,42182759,USDA-NVSL,2025,...,2025-06-06 00:57:17,2025-06-04 14:25:01,1,25-015677-003,SRP503016,NaN,"MILK, BULK TANK",SRS25266463,False,NaN
19047,SRR33830448,WGS,148.30,110812426,PRJNA1102327,SAMN48895424,Viral,43738503,USDA-NVSL,2025,...,2025-06-06 00:57:17,2025-06-04 14:24:55,1,25-015677-002,SRP503016,NaN,"MILK, BULK TANK",SRS25266462,False,NaN
19048,SRR33830449,WGS,131.52,67358891,PRJNA1102327,SAMN48895415,Viral,26336550,USDA-NVSL,2024,...,2025-06-06 00:57:16,2025-06-04 14:24:53,1,24-036379-001-tile,SRP503016,NaN,"MILK, BULK TANK",SRS25266461,False,NaN


In [ ]:
# Collapse dataset to only sequences in original dataset AND without dates OR states

for key in original_fasta_dfs:
    og_df = original_fasta_dfs[key]
    # print(og_df)
    og_df["Unknown_States"] = og_df["Geo_Location"].apply(lambda x: 1 if x == "USA" else 0)
    og_df["Unknown_Dates"] = og_df["Date Collected"].apply(lambda x: 1 if x == "2023" or x == "2024" or x == "2025" else 0)
    og_df["Update_Needed"] = og_df["Unknown_States"] + og_df["Unknown_Dates"]
    # og_df["Identifier"] = ""
    og_df_update_needed = og_df[og_df["Update_Needed"] > 0]
    for isolate in og_df_update_needed["Isolate_Id"].values:
        # print(isolate)
        # for new_df in metadata:
        if isolate in metadata["isolate"].values:
            # print(isolate)
            og_df_update_needed.at[og_df_update_needed[og_df_update_needed["Isolate_Id"] == isolate].index[0], "Geo_Location"] = metadata.loc[metadata[metadata["isolate"] == isolate].index[0], "geo_loc_name"]
            og_df_update_needed.at[og_df_update_needed[og_df_update_needed["Isolate_Id"] == isolate].index[0], "Date Collected"] = metadata[metadata["isolate"] == isolate]["BioSample"].apply(lambda x: search_collection_date(x, metadata)).values[0]
            og_df_update_needed.at[og_df_update_needed[og_df_update_needed["Isolate_Id"] == isolate].index[0], "Identifier"] = metadata.loc[metadata[metadata["isolate"] == isolate].index[0], "Run"]

    print(og_df_update_needed)

    # new_df = og_df.merge(og_df_update_needed, how="left")
    # new_df = pd.concat([og_df_update_needed, og_df]).drop_duplicates('Isolate_Id', keep="first")
    new_df = og_df.set_index('Header')
    new_df.update(og_df_update_needed.set_index('Header'))
    new_df = new_df.reset_index()

    print(new_df)
    # break 

    original_fasta_dfs[key] = new_df

                                                 Header     Isolate_Id  \
0     A/AMERICAN_BLACK_DUCK/USA/25-002460-001/2025|H...  25-002460-001   
1     A/AMERICAN_BLACK_DUCK/USA/25-002634-002/2025|H...  25-002634-002   
6     A/American_green-winged_teal/USA/IZ24_0776/202...      IZ24_0776   
7     A/American_green-winged_teal/USA/IZ24_0777/202...      IZ24_0777   
9     A/BALD_EAGLE/USA/25-006702-001/2025|H5N1|USA|2...  25-006702-001   
...                                                 ...            ...   
2233  A/wood_duck/IL/001096-002/2025|H5N1|USA-IL|202...     001096-002   
2236  A/wood_duck/USA/003927-016/2025|H5N1|USA|2025|...     003927-016   
2237  A/wood_duck/USA/005011-001/2025|H5N1|USA|2025|...     005011-001   
2238  A/wood_duck/USA/007113-001/2025|H5N1|USA|2025|...     007113-001   
2239  A/wood_duck/USA/038225-003/2024|H5N1|USA|2024|...     038225-003   

                                         Isolate_Name Subtype Geo_Location  \
0        A/AMERICAN_BLACK_DUCK/US

## NCBI Virus

In [ ]:
os.chdir(downloads_ncbi_virus + date_range + "/")

# Read metadata
ncbi_metadata = pd.read_csv("sequences.csv")

# # Integrate genotypes
# os.chdir(downloads_ncbi_virus + "11-01-2021--04-14-2025/")
# apr_output = pd.read_csv("output.tsv", delimiter="\t")

# os.chdir(downloads_ncbi_virus + "04-14-2025--05-14-2025/")
# may_output = pd.read_csv("output.tsv", delimiter="\t")

print(ncbi_metadata)

        Accession      Organism_Name GenBank_RefSeq         Assembly  \
0      PV635323.1  Influenza A virus        GenBank  GCA_050380315.1   
1      PV635324.1  Influenza A virus        GenBank  GCA_050380315.1   
2      PV635325.1  Influenza A virus        GenBank  GCA_050380315.1   
3      PV635326.1  Influenza A virus        GenBank  GCA_050380315.1   
4      PV635327.1  Influenza A virus        GenBank  GCA_050380315.1   
...           ...                ...            ...              ...   
65449  PP577943.1  Influenza A virus        GenBank  GCA_039465435.1   
65450  PP577944.1  Influenza A virus        GenBank  GCA_039465435.1   
65451  PP577945.1  Influenza A virus        GenBank  GCA_039465435.1   
65452  PP577946.1  Influenza A virus        GenBank  GCA_039465435.1   
65453  PP577947.1  Influenza A virus        GenBank  GCA_039465435.1   

      SRA_Accession                                         Submitters  \
0               NaN  Jahid,M.J., Bowman,A.S., Nolting,J.M., R

In [ ]:
# Search for dates and states based on isolate

for key in original_fasta_dfs:
    og_df = original_fasta_dfs[key]
    # print(og_df)
    og_df["Unknown_States"] = og_df["Geo_Location"].apply(lambda x: 1 if x == "USA" else 0)
    og_df["Unknown_Dates"] = og_df["Date Collected"].apply(lambda x: 1 if x == "2023" or x == "2024" or x == "2025" else 0)

    # og_df["Unknown_Dates"] = og_df["Date Collected"].apply(dateutil.parser.parse).apply(lambda x: 1 if x == dateutil.parser.parse("2023-01-01") or x == dateutil.parser.parse("2024-01-01") or x == dateutil.parser.parse("2025-01-01") else 0)
    og_df["Update_Needed"] = og_df["Unknown_States"] + og_df["Unknown_Dates"]
    # og_df["Identifier"] = ""
    og_df_update_needed = og_df[og_df["Update_Needed"] > 0]
    # print(og_df_update_needed)
    for isolate in og_df_update_needed["Isolate_Id"].values:
        # print(isolate)
        # for new_df in ncbi_metadata:
            # print(new_df)
        if isolate in ncbi_metadata["Isolate"].values:
            print(isolate)
            og_df_update_needed.at[og_df_update_needed[og_df_update_needed["Isolate_Id"] == isolate].index[0], "Geo_Location"] = ncbi_metadata.loc[ncbi_metadata[ncbi_metadata["Isolate"] == isolate].index[0], "Geo_Location"] # .replace(": ", "-")
            og_df_update_needed.at[og_df_update_needed[og_df_update_needed["Isolate_Id"] == isolate].index[0], "Date Collected"] = ncbi_metadata.loc[ncbi_metadata[ncbi_metadata["Isolate"] == isolate].index[0], "Collection_Date"]
            og_df_update_needed.at[og_df_update_needed[og_df_update_needed["Isolate_Id"] == isolate].index[0], "Identifier"] = ncbi_metadata.loc[ncbi_metadata[ncbi_metadata["Isolate"] == isolate].index[0], "SRA_Accession"]

    # new_df = og_df.merge(og_df_update_needed, how="left")
    # new_df = pd.concat([og_df_update_needed, og_df]).drop_duplicates('Isolate_Id', keep="first")

    new_df = og_df.set_index('Header')
    new_df.update(og_df_update_needed.set_index('Header'))
    new_df = new_df.reset_index()

    print(new_df)
    # break 

    original_fasta_dfs[key] = new_df

IZ24_0776
IZ24_0777
IZ24_0675
IZ24_0555
IZ24_0566
                                                 Header     Isolate_Id  \
0     A/AMERICAN_BLACK_DUCK/USA/25-002460-001/2025|H...  25-002460-001   
1     A/AMERICAN_BLACK_DUCK/USA/25-002634-002/2025|H...  25-002634-002   
2     A/American_Crow/BC/AIVPHL-2662/2024|H5N1|Canad...    AIVPHL-2662   
3     A/American_Crow/BC/AIVPHL-2850/2024|H5N1|Canad...    AIVPHL-2850   
4     A/American_Kestrel/BC/AIVPHL-2866/2024|H5N1|Ca...    AIVPHL-2866   
...                                                 ...            ...   
2235  A/wood_duck/Tennessee/79/2024|H5N1|USA-TN|2024...             79   
2236  A/wood_duck/USA/003927-016/2025|H5N1|USA|2025|...     003927-016   
2237  A/wood_duck/USA/005011-001/2025|H5N1|USA|2025|...     005011-001   
2238  A/wood_duck/USA/007113-001/2025|H5N1|USA|2025|...     007113-001   
2239  A/wood_duck/USA/038225-003/2024|H5N1|USA|2024|...     038225-003   

                                      Isolate_Name Subtype Ge

# Put it all together

In [ ]:

for key in original_fasta_dfs:
    df = original_fasta_dfs[key]
    df["Identifier"] = df["Identifier"].apply(lambda x: "unknown" if x != x else x) # Put "unknown" if NaN
    df["full_header"] = ">" + df["Identifier"] + "|" + df["Isolate_Name"] + "|" + df["Subtype"] + "|" + df["Geo_Location"] + "|" + df["Date Collected"] + "|" + df["Host_Type"] + "|" + df["Genotype"] + "\n"
    df["sequence"] = df["Sequence"]

    df_to_fasta(df, key.split("/")[-1][:-6] + "_updated_" + update_date + ".fasta", complete)